In [55]:
# load the necessary python modules
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
import random
import gymnasium as gym
import sys
import warnings
import time

# ignore warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")
# initialize the nchain environment
# import the frozen lake gym environment
name = 'FrozenLake-v1'
env = gym.make(name, is_slippery=False) # warning: setting slippery=True results in very complex environment dynamics where the optimal solution does not make sense to humans!
seed=742

obs, info = env.reset(seed=seed)
env.action_space.seed(seed)
rng = np.random.default_rng(seed)

# lets examine it
print('action space: ' + str(env.action_space))
print('reward range: ' + str(env.reward_range) if hasattr(env, "reward_range") else "(0,1)")
print('observation space: ' + str(env.observation_space))
env.render()



action space: Discrete(4)
reward range: (0, 1)
observation space: Discrete(16)


In [56]:
class Qagent(object):
    """
    Implementation of a Q-learning Algorithm
    """

    def __init__(
        self,
        action_size: int,
        state_size: int,
        learning_parameters: dict,
        exploration_parameters: dict,
        name: str = "agent",
        color: str = "r",
    ) -> None:
        """ initialize the q-learning agent

        Args:
            action_size (int): number of actions the agent can take
            state_size (int): numbe of states the env has
            learning_parameters (dict): learning paramters of the agent
            exploration_parameters (dict): exploration paramters for the agent
            name (str, optional):  set the name of the Q-Agent. Defaults to "agent".
            color (str, optional): set the color of the agent for plotting. Defaults to "r".
        """
        self.name = name
        self.color = color

        self.action_size = action_size
        self.state_size = state_size
        self.qtable = np.zeros((state_size, action_size))

        self.learning_rate = learning_parameters["learning_rate"]
        self.gamma = learning_parameters["gamma"]

        self.epsilon = exploration_parameters["epsilon"]
        self.max_epsilon = exploration_parameters["max_epsilon"]
        self.min_epsilon = exploration_parameters["min_epsilon"]
        self.decay_rate = exploration_parameters["decay_rate"]

    def update_qtable(
        self, state: int, new_state: int, action: int, reward: int, done: bool
    ) -> None:
        """
        update the q-table: Q(s,a) = Q(s,a) + lr  * [R(s,a) + gamma * max Q(s',a') - Q (s,a)]

        Args:
          state (int): current state of the environment
          new_state (int): new state of the environment
          action (int): current action taken by agent
          reward (int): current reward received from env
          done (boolean): variable indicating if env is done
        """
        new_qvalue = reward + self.gamma * np.max(self.qtable[new_state, :]) * (not done)

        self.qtable[state, action] = self.qtable[state, action] + self.learning_rate * (new_qvalue - self.qtable[state, action])

    def update_epsilon(self, episode: int) -> None:
        """
        reduce epsilon, exponential decay
        e = min_e + (max_e -min_e) * exp(-deacy_rate*episode)

        Args:
          episode (int): number of episode
        """
        self.epsilon = self.min_epsilon + (self.max_epsilon - self.min_epsilon) * np.exp(
            -self.decay_rate * episode
        )

    def get_action(self, state: int) -> int:
        """
        select action e-greedy.exploration-exploitation trade-off:
        - exploitation, max value for given state
        - exploration, random choice

        Args:
          state (int): current state of the environment/agent

        Returns:
          action (int): action that the agent will take in the next step
        """
        if random.uniform(0, 1) >= self.epsilon:
            action = np.argmax(self.qtable[state, :])
        else:
            action = np.random.choice(self.action_size)
        return action

    def __str__(self, tablefmt="fancy_grid") -> None:
        """plot the q-table. Generate the table in fancy format.
        """
        headers = [f"Action {action}" for action in range(self.action_size)]
        showindex = [f"State {state}" for state in range(self.state_size)]
        table = tabulate(self.qtable, headers=headers, showindex=showindex, tablefmt="fancy_grid")
        return f"{self.name}\n{table}"

In [57]:
def learn_to_play(agent: Qagent, max_game_steps: int = 10, total_episodes: int = 1000) -> Qagent:
    """
    implementation of the q-learning algorithm, here the q-table values are calculated

    Args:
      max_game_steps (int): number of stepts an agent can take, before the environment is reset
      total_episodes (int): total of training episodes (the number of trials a agent can do)
    """

    rewards = np.zeros(total_episodes)
    epsilons = np.zeros(total_episodes)
    last_states = np.zeros(total_episodes)
    q_averages = np.zeros(total_episodes)

    start = time.time()

    for episode in range(total_episodes):

        state, _ = env.reset()
        game_rewards = 0

        # for each episode loop over the max number of steps that are possible
        # take an action and observe the outcome state (new_state), reward and stopping criterion
        for step in range(max_game_steps):

            action = agent.get_action(state)
            new_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            agent.update_qtable(state, new_state, action, reward, done)
            state = new_state
            game_rewards += reward

            if done == True:
                break

        rewards[episode] = game_rewards
        last_states[episode] = state
        epsilons[episode] = agent.epsilon
        q_averages[episode] = np.sum(agent.qtable)

        # reduce epsilon, for exploration-exploitation tradeoff
        agent.update_epsilon(episode)

        if episode % 300 == 0:
            elapsed_time = round((time.time() - start), 1)
            print(f"elapsed time [sec]: {elapsed_time}, episode: {episode}")

    agent.rewards = rewards
    agent.last_states = last_states
    agent.epsilons = epsilons
    agent.q_averages = q_averages
    return agent
action_size = env.action_space.n
state_size = env.observation_space.n

# Set the training parameters

max_game_steps = 1000  # Set number of stepts an agent can take, before the environment is reset, 
total_episodes = 10000  # Set total of training episodes (the number of trials a agent can do)    


In [58]:
name = 'Smart Agent 1 - the agent explores and takes future rewards into accountt'
color = "orange"

learning_parameters = {
    'learning_rate': 0.8,
    'gamma': 0.9 
}  
exploration_parameters = {
    'epsilon': 1,
    'max_epsilon': 1,
    'min_epsilon': 0.0,
    'decay_rate': 0.008
} 

q_agent_1 = Qagent(action_size, state_size, learning_parameters, exploration_parameters, name, color)
q_agent_1 = learn_to_play(q_agent_1, max_game_steps=max_game_steps, total_episodes=total_episodes)

elapsed time [sec]: 0.0, episode: 0
elapsed time [sec]: 0.1, episode: 300
elapsed time [sec]: 0.1, episode: 600
elapsed time [sec]: 0.2, episode: 900
elapsed time [sec]: 0.2, episode: 1200
elapsed time [sec]: 0.2, episode: 1500
elapsed time [sec]: 0.3, episode: 1800
elapsed time [sec]: 0.3, episode: 2100
elapsed time [sec]: 0.3, episode: 2400
elapsed time [sec]: 0.3, episode: 2700
elapsed time [sec]: 0.4, episode: 3000
elapsed time [sec]: 0.4, episode: 3300
elapsed time [sec]: 0.4, episode: 3600
elapsed time [sec]: 0.5, episode: 3900
elapsed time [sec]: 0.5, episode: 4200
elapsed time [sec]: 0.5, episode: 4500
elapsed time [sec]: 0.6, episode: 4800
elapsed time [sec]: 0.6, episode: 5100
elapsed time [sec]: 0.6, episode: 5400
elapsed time [sec]: 0.7, episode: 5700
elapsed time [sec]: 0.7, episode: 6000
elapsed time [sec]: 0.7, episode: 6300
elapsed time [sec]: 0.7, episode: 6600
elapsed time [sec]: 0.8, episode: 6900
elapsed time [sec]: 0.8, episode: 7200
elapsed time [sec]: 0.8, episod

In [59]:
name = 'Greedy Agent 2 - the agent cares only about immediate rewards (small gamma)'
color =  "m"

learning_parameters = {
    'learning_rate': 0.8,
    'gamma': 0.01
}   
exploration_parameters = {
    'epsilon': 1,
    'max_epsilon': 0.5,
    'min_epsilon': 0.0,
    'decay_rate': 0.008
} 

q_agent_2 = Qagent(action_size, state_size, learning_parameters, exploration_parameters, name, color)
q_agent_2 = learn_to_play(q_agent_2, max_game_steps=max_game_steps, total_episodes=total_episodes)

elapsed time [sec]: 0.0, episode: 0
elapsed time [sec]: 0.3, episode: 300
elapsed time [sec]: 0.7, episode: 600
elapsed time [sec]: 1.2, episode: 900
elapsed time [sec]: 1.6, episode: 1200
elapsed time [sec]: 2.0, episode: 1500
elapsed time [sec]: 2.5, episode: 1800
elapsed time [sec]: 2.9, episode: 2100
elapsed time [sec]: 3.4, episode: 2400
elapsed time [sec]: 3.8, episode: 2700
elapsed time [sec]: 4.2, episode: 3000
elapsed time [sec]: 4.7, episode: 3300
elapsed time [sec]: 5.1, episode: 3600
elapsed time [sec]: 5.6, episode: 3900
elapsed time [sec]: 6.0, episode: 4200
elapsed time [sec]: 6.5, episode: 4500
elapsed time [sec]: 6.9, episode: 4800
elapsed time [sec]: 7.3, episode: 5100
elapsed time [sec]: 7.8, episode: 5400
elapsed time [sec]: 8.2, episode: 5700
elapsed time [sec]: 8.7, episode: 6000
elapsed time [sec]: 9.1, episode: 6300
elapsed time [sec]: 9.6, episode: 6600
elapsed time [sec]: 10.0, episode: 6900
elapsed time [sec]: 10.5, episode: 7200
elapsed time [sec]: 10.9, epi

In [60]:
name = "Shy Agent 3 - the agent doesn't explore the environment (small epsilon)"
color = "b"

learning_parameters = {
    'learning_rate': 0.8,
    'gamma': 0.9
} 
exploration_parameters = {
    'epsilon': 1,
    'max_epsilon': 0.2,
    'min_epsilon': 0.0,
    'decay_rate': 0.5
} 

q_agent_3 = Qagent(action_size, state_size, learning_parameters, exploration_parameters, name, color)
q_agent_3 = learn_to_play(q_agent_3, max_game_steps=max_game_steps, total_episodes=total_episodes)

elapsed time [sec]: 0.0, episode: 0
elapsed time [sec]: 0.4, episode: 300
elapsed time [sec]: 0.9, episode: 600
elapsed time [sec]: 1.3, episode: 900
elapsed time [sec]: 1.8, episode: 1200
elapsed time [sec]: 2.2, episode: 1500
elapsed time [sec]: 2.7, episode: 1800
elapsed time [sec]: 3.1, episode: 2100
elapsed time [sec]: 3.6, episode: 2400
elapsed time [sec]: 4.0, episode: 2700
elapsed time [sec]: 4.5, episode: 3000
elapsed time [sec]: 4.9, episode: 3300
elapsed time [sec]: 5.4, episode: 3600
elapsed time [sec]: 5.8, episode: 3900
elapsed time [sec]: 6.3, episode: 4200
elapsed time [sec]: 6.7, episode: 4500
elapsed time [sec]: 7.2, episode: 4800
elapsed time [sec]: 7.6, episode: 5100
elapsed time [sec]: 8.0, episode: 5400
elapsed time [sec]: 8.5, episode: 5700
elapsed time [sec]: 8.9, episode: 6000
elapsed time [sec]: 9.4, episode: 6300
elapsed time [sec]: 9.8, episode: 6600
elapsed time [sec]: 10.3, episode: 6900
elapsed time [sec]: 10.8, episode: 7200
elapsed time [sec]: 11.2, epi

In [61]:
from IPython.display import Image

In [ ]:
class VisualizePlays:
    """Visualize the training progress of multiple Q-learning agents"""
    
    def __init__(self, *agents):
        """Initialize with one or more agents"""
        self.agents = agents
    
    def plot(self):
        """Plot comparison of agents' performance"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Q-Learning Agents Comparison', fontsize=16, fontweight='bold')
        
        # Plot 1: Rewards over episodes
        ax1 = axes[0, 0]
        for agent in self.agents:
            # Smooth the rewards with a moving average
            window_size = 100
            smoothed_rewards = np.convolve(agent.rewards, np.ones(window_size)/window_size, mode='valid')
            ax1.plot(smoothed_rewards, label=agent.name, color=agent.color, linewidth=2)
        ax1.set_xlabel('Episode', fontsize=12)
        ax1.set_ylabel('Average Reward (smoothed)', fontsize=12)
        ax1.set_title('Learning Progress: Rewards Over Time', fontsize=13, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Epsilon decay
        ax2 = axes[0, 1]
        for agent in self.agents:
            ax2.plot(agent.epsilons, label=agent.name, color=agent.color, linewidth=2)
        ax2.set_xlabel('Episode', fontsize=12)
        ax2.set_ylabel('Epsilon Value', fontsize=12)
        ax2.set_title('Exploration Rate (Epsilon) Decay', fontsize=13, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Q-table sum evolution
        ax3 = axes[1, 0]
        for agent in self.agents:
            ax3.plot(agent.q_averages, label=agent.name, color=agent.color, linewidth=2)
        ax3.set_xlabel('Episode', fontsize=12)
        ax3.set_ylabel('Sum of Q-values', fontsize=12)
        ax3.set_title('Q-Table Evolution (Sum of All Q-values)', fontsize=13, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Success rate (reaching goal state)
        ax4 = axes[1, 1]
        for agent in self.agents:
            # Calculate success rate (state 15 is the goal in FrozenLake)
            window_size = 100
            successes = (agent.rewards > 0).astype(int)  # 1 if reached goal, 0 otherwise
            success_rate = np.convolve(successes, np.ones(window_size)/window_size, mode='valid') * 100
            ax4.plot(success_rate, label=agent.name, color=agent.color, linewidth=2)
        ax4.set_xlabel('Episode', fontsize=12)
        ax4.set_ylabel('Success Rate (%)', fontsize=12)
        ax4.set_title('Success Rate (Reaching Goal)', fontsize=13, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\n" + "="*80)
        print("SUMMARY STATISTICS")
        print("="*80)
        for agent in self.agents:
            total_rewards = np.sum(agent.rewards)
            avg_reward = np.mean(agent.rewards)
            success_count = np.sum(agent.rewards > 0)
            success_rate = (success_count / len(agent.rewards)) * 100
            final_epsilon = agent.epsilons[-1]
            
            print(f"\n{agent.name}")
            print("-" * 80)
            print(f"  Total Rewards:        {total_rewards:.2f}")
            print(f"  Average Reward:       {avg_reward:.4f}")
            print(f"  Success Count:        {success_count}/{len(agent.rewards)}")
            print(f"  Success Rate:         {success_rate:.2f}%")
            print(f"  Final Epsilon:        {final_epsilon:.4f}")
            print(f"  Final Q-table Sum:    {agent.q_averages[-1]:.2f}")

In [ ]:
plays = VisualizePlays(q_agent_1, q_agent_2, q_agent_3)
plays.plot()

NameError: name 'VisualizePlays' is not defined

In [63]:
#!pip install helper_functions

In [71]:
print(q_agent_1)

Smart Agent 1 - the agent explores and takes future rewards into accountt
╒══════════╤════════════╤════════════╤════════════╤════════════╕
│          │   Action 0 │   Action 1 │   Action 2 │   Action 3 │
╞══════════╪════════════╪════════════╪════════════╪════════════╡
│ State 0  │   0.531441 │   0.59049  │   0.478288 │   0.531441 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 1  │   0.531441 │   0        │   0        │   0        │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 2  │   0        │   0        │   0        │   0        │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 3  │   0        │   0        │   0        │   0        │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 4  │   0.590488 │   0.6561   │   0        │   0.530498 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 5  │   0        │   0        │   0        │   0        │
├──────────┼────

In [67]:
print(q_agent_2)

Greedy Agent 2 - the agent cares only about immediate rewards (small gamma)
╒══════════╤════════════╤════════════╤════════════╤════════════╕
│          │   Action 0 │   Action 1 │   Action 2 │   Action 3 │
╞══════════╪════════════╪════════════╪════════════╪════════════╡
│ State 0  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 1  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 2  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 3  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 4  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 5  │          0 │          0 │          0 │          0 │
├──────────┼──

In [72]:
print(q_agent_3)

Shy Agent 3 - the agent doesn't explore the environment (small epsilon)
╒══════════╤════════════╤════════════╤════════════╤════════════╕
│          │   Action 0 │   Action 1 │   Action 2 │   Action 3 │
╞══════════╪════════════╪════════════╪════════════╪════════════╡
│ State 0  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 1  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 2  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 3  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 4  │          0 │          0 │          0 │          0 │
├──────────┼────────────┼────────────┼────────────┼────────────┤
│ State 5  │          0 │          0 │          0 │          0 │
├──────────┼──────